In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_predict, TunedThresholdClassifierCV,StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, make_scorer, fbeta_score
from joblib import dump, load
import numpy as np

In [4]:
data = pd.read_csv("Data/processed_data")
random_forest = load("models/RandomForest.joblib")
boosting = load("models/Boosting.joblib")

In [5]:
X = data[["annual_income_ru", "loan_ammount_ru", "int_rate_ru", "DTI","home_ownership", "purpose"]] 
Y = data["is_loss"]

X["purpose"] = X["purpose"].astype("category")
X["home_ownership"] = X["home_ownership"].astype("category")
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, random_state=42, stratify=Y,
                                                    train_size=0.85)

C:\Users\PC12\AppData\Local\Temp\ipykernel_12972\2539622627.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X["purpose"] = X["purpose"].astype("category")
C:\Users\PC12\AppData\Local\Temp\ipykernel_12972\2539622627.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X["home_ownership"] = X["home_ownership"].astype("category")


In [11]:
def get_meta_features(X, y, model1, model2, cv):
    meta_X = pd.DataFrame(index=X.index)
    # Для первой модели
    fold_preds1 = cross_val_predict(
        model1, X, y, cv=cv, 
        method='predict_proba', 
        n_jobs=-1
    )[:, 1]
    meta_X['model1_proba'] = fold_preds1
    
    # Для второй модели
    fold_preds2 = cross_val_predict(
        model2, X, y, cv=cv, 
        method='predict_proba', 
        n_jobs=-1
    )[:, 1]
    meta_X['model2_proba'] = fold_preds2
    
    return meta_X

# Создаем мета-признаки для тренировки
X_meta_train = get_meta_features(X_train, Y_train, random_forest, boosting, 10)
X_meta_train

,model1_proba,model2_proba
13836,0.399362,0.337022
22927,0.708629,0.665883
25687,0.463877,0.413907
3452,0.462391,0.401795
3508,0.645366,0.619049
...,...,...
26470,0.541784,0.499231
38234,0.427743,0.375134
663,0.200514,0.133727
28864,0.340624,0.268120


In [18]:
meta_model = TunedThresholdClassifierCV(estimator=LogisticRegression(random_state=42),
                                        scoring="f1", cv=StratifiedKFold(shuffle=True,random_state=42,n_splits=10),
                                        n_jobs = -1)
meta_model.fit(X_meta_train, Y_train)
print(meta_model.best_threshold_)
print(meta_model.best_score_)

0.15868967624197738
0.3214377256807994


In [17]:
# rf_test_proba = random_forest.predict_proba(X_test)[:, 1]
# xgb_test_proba = boosting.predict_proba(X_test)[:, 1]

# # Собираем мета-признаки для теста
# X_meta_test = pd.DataFrame({
#     'model1_proba': rf_test_proba,
#     'model2_proba': xgb_test_proba
# })

# # Финальное предсказание
# final_proba = meta_model.predict_proba(X_meta_test)[:, 1]
# prediction  =meta_model.predict(X_meta_test)
# print(f1_score(Y_test, prediction))

IndexError: list index out of range